In [1]:
!pip install -qq git+https://github.com/csebuetnlp/normalizer
!pip install -Uq bitsandbytes
!pip install -Uq transformers peft


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 4.1 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 79.3 MB/s eta 0:00:00:00:01:01


In [2]:
import warnings, os, re, unicodedata, random
import torch
import torch.nn as nn
from collections import OrderedDict
from typing import Optional, Union, Tuple
from PIL import Image
from transformers import (
    Blip2Processor,
    Blip2PreTrainedModel,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoConfig,
    BitsAndBytesConfig,
)
from transformers.models.blip_2.modeling_blip_2 import Blip2ForConditionalGenerationModelOutput
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from normalizer import normalize

warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def bangla_postprocess(text):
    if not text or not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\u200b', '').replace('\u200c', '').replace('\u200d', '').replace('\ufeff', '')
    text = normalize(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Imports done")


Imports done


In [3]:
# ═══════════════════════════════════════════════════════
# BanglaBLIP model class — must be identical to training
# ═══════════════════════════════════════════════════════

class BanglaBLIP(Blip2PreTrainedModel):
    def __init__(self,
                 blip_pretrained="Salesforce/blip2-flan-t5-xl",
                 bangla_lm_pretrained="csebuetnlp/banglat5",
                 load_8bit=True,
                 freeze_vit=True,
                 freeze_qformer=True,
                 freeze_lm=True,
                 freeze_projection=False,
                 use_lora=False,
                 lora_alpha=16,
                 lora_r=8,
                 lora_dropout=0.05,
                 lora_bias="none",
                 lora_checkpoint=None):

        from transformers import Blip2ForConditionalGeneration

        blip2_model = Blip2ForConditionalGeneration.from_pretrained(
            blip_pretrained, torch_dtype=torch.float16,
        )
        config = blip2_model.config
        bangla_config = AutoConfig.from_pretrained(bangla_lm_pretrained)
        config.text_config = bangla_config
        super().__init__(config)

        self.vision_model  = blip2_model.vision_model
        self.qformer       = blip2_model.qformer
        self.query_tokens  = blip2_model.query_tokens

        self.language_projection = nn.Sequential(
            nn.Linear(self.qformer.config.hidden_size, bangla_config.d_model),
            nn.GELU(),
            nn.Linear(bangla_config.d_model, bangla_config.d_model)
        )
        self.language_projection_ln = nn.LayerNorm(bangla_config.d_model)

        del blip2_model.language_model
        del blip2_model
        torch.cuda.empty_cache()

        rank = int(os.environ.get("LOCAL_RANK", 0))
        self.llm_cast_dtype = torch.bfloat16

        if isinstance(load_8bit, str) and load_8bit == "4bit":
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=self.llm_cast_dtype,
            )
            self.language_model = AutoModelForSeq2SeqLM.from_pretrained(
                bangla_lm_pretrained, dtype="auto",
                quantization_config=bnb_config, device_map={"": rank},
            )
        elif load_8bit:
            bnb_config = BitsAndBytesConfig(load_in_8bit=True)
            self.language_model = AutoModelForSeq2SeqLM.from_pretrained(
                bangla_lm_pretrained, dtype="auto",
                quantization_config=bnb_config, device_map={"": rank},
            )
        else:
            self.language_model = AutoModelForSeq2SeqLM.from_pretrained(
                bangla_lm_pretrained, dtype="auto", device_map={"": rank},
            )

        for param in self.vision_model.parameters():
            param.requires_grad = False

        if use_lora:
            self.language_model = prepare_model_for_kbit_training(
                self.language_model, use_gradient_checkpointing=False
            )
            if lora_checkpoint and os.path.exists(lora_checkpoint):
                self.language_model = PeftModel.from_pretrained(
                    self.language_model, lora_checkpoint
                )
            else:
                target_modules = ["q", "k", "v", "o", "wi_0", "wi_1", "wo"]
                lora_config = LoraConfig(
                    r=lora_r, lora_alpha=lora_alpha,
                    target_modules=target_modules,
                    lora_dropout=lora_dropout,
                    bias=lora_bias, task_type="SEQ_2_SEQ_LM"
                )
                self.language_model = get_peft_model(self.language_model, lora_config)

        self._use_lora = bool(use_lora)

    def get_input_embeddings(self):
        return self.language_model.get_input_embeddings()

    def set_input_embeddings(self, value):
        self.language_model.set_input_embeddings(value)

    @torch.no_grad()
    def generate(self, pixel_values, input_ids=None, attention_mask=None,
                 **generate_kwargs):
        orig_batch_size = pixel_values.shape[0]

        image_embeds = self.vision_model(pixel_values, return_dict=True).last_hidden_state
        image_attention_mask = torch.ones(
            image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device
        )
        query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
        query_output = self.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True,
        ).last_hidden_state

        language_model_inputs = self.language_projection(query_output)
        language_model_inputs = self.language_projection_ln(language_model_inputs)
        language_attention_mask = torch.ones(
            language_model_inputs.size()[:-1], dtype=torch.long,
            device=language_model_inputs.device
        )

        if input_ids is None:
            input_ids = (
                torch.LongTensor([[self.config.text_config.bos_token_id]])
                .repeat(orig_batch_size, 1).to(image_embeds.device)
            )
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        attention_mask = torch.cat([language_attention_mask, attention_mask], dim=1)

        with torch.amp.autocast("cuda", dtype=self.llm_cast_dtype):
            inputs_embeds = self.language_model.get_input_embeddings()(input_ids)
            inputs_embeds = torch.cat(
                [language_model_inputs, inputs_embeds.to(language_model_inputs.device)],
                dim=1
            )
            outputs = self.language_model.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                **generate_kwargs,
            )
        return outputs

print("BanglaBLIP class defined")


BanglaBLIP class defined


In [11]:
# ═══════════════════════════════════════════════════════
# INFERENCE CONFIG
# Change CHECKPOINT_DIR to whichever experiment you want:
#   /kaggle/input/<your-dataset>/LoRA_r16/final_model
#   /kaggle/input/<your-dataset>/LoRA_r32/final_model
#   /kaggle/input/<your-dataset>/LoRA_r64/final_model  ← best results
# ═══════════════════════════════════════════════════════

CHECKPOINT_DIR = "/kaggle/input/datasets/ibtida01/banglablip-outputs/LoRA_r64/final_model"
IMAGES_DIR      = "/kaggle/input/datasets/adityajn105/flickr8k/Images"
CAPTIONS_CSV    = "/kaggle/input/datasets/saifsust/bancap/BAN-Cap_captiondata.csv"
OUTPUT_CSV      = "/kaggle/working/inference_results.csv"

BLIP_CHECKPOINT  = "Salesforce/blip2-flan-t5-xl"
BANGLA_MODEL_ID  = "csebuetnlp/banglat5"

# LoRA config must match training exactly
LORA_R       = 64
LORA_ALPHA   = 128
LORA_DROPOUT = 0.05

NUM_SAMPLES  = 50    # how many images to caption (None = all test images)

MAX_GEN_LENGTH    = 64
NUM_BEAMS         = 5
LENGTH_PENALTY    = 0.4
REPETITION_PENALTY = 1.5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_DIR}")


Device: cuda
Checkpoint: /kaggle/input/datasets/ibtida01/banglablip-outputs/LoRA_r64/final_model


In [27]:
import safetensors.torch as st
import os

# Step 1: কী কী keys saved checkpoint এ আছে দেখো
ckpt_path = os.path.join(CHECKPOINT_DIR, "model.safetensors")
saved_weights = st.load_file(ckpt_path)

print("=== SAVED CHECKPOINT KEYS (first 20) ===")
for k in list(saved_weights.keys())[:20]:
    print(f"  {k}  →  {saved_weights[k].shape}")

print(f"\nTotal saved keys: {len(saved_weights)}")

# Step 2: Model এর current keys দেখো
print("\n=== MODEL KEYS (first 20) ===")
model_keys = list(model.state_dict().keys())
for k in model_keys[:20]:
    print(f"  {k}")

print(f"\nTotal model keys: {len(model_keys)}")

# Step 3: Overlap দেখো
saved_set = set(saved_weights.keys())
model_set = set(model_keys)
matched = saved_set & model_set
print(f"\nMatched keys: {len(matched)}")
print(f"In checkpoint but NOT in model: {len(saved_set - model_set)}")
print(f"In model but NOT in checkpoint: {len(model_set - saved_set)}")

=== SAVED CHECKPOINT KEYS (first 20) ===
  language_projection.0.bias  →  torch.Size([768])
  language_projection.0.weight  →  torch.Size([768, 768])
  language_projection.2.bias  →  torch.Size([768])
  language_projection.2.weight  →  torch.Size([768, 768])
  language_projection_ln.bias  →  torch.Size([768])
  language_projection_ln.weight  →  torch.Size([768])
  qformer.encoder.layer.0.attention.attention.key.bias  →  torch.Size([768])
  qformer.encoder.layer.0.attention.attention.key.weight  →  torch.Size([768, 768])
  qformer.encoder.layer.0.attention.attention.query.bias  →  torch.Size([768])
  qformer.encoder.layer.0.attention.attention.query.weight  →  torch.Size([768, 768])
  qformer.encoder.layer.0.attention.attention.value.bias  →  torch.Size([768])
  qformer.encoder.layer.0.attention.attention.value.weight  →  torch.Size([768, 768])
  qformer.encoder.layer.0.attention.output.LayerNorm.bias  →  torch.Size([768])
  qformer.encoder.layer.0.attention.output.LayerNorm.weight  →  

In [10]:
# import os

# base = "/kaggle/input/datasets/ibtida01"
# for root, dirs, files in os.walk(base):
#     level = root.replace(base, '').count(os.sep)
#     indent = '  ' * level
#     print(f"{indent}{os.path.basename(root)}/")
#     for f in files:
#         print(f"  {indent}{f}")

ibtida01/
  banglablip-outputs/
    LoRA_r64/
      experiment_summary.json
      test_results.csv
      stage1_warmup/
        checkpoint-506/
          config.json
          trainer_state.json
          training_args.bin
          tokenizer.json
          tokenizer_config.json
          scheduler.pt
          model.safetensors
          optimizer.pt
          rng_state.pth
      final_model/
        config.json
        tokenizer.json
        tokenizer_config.json
        model.safetensors
      stage1_checkpoint/
        config.json
        model.safetensors
      stage2_realignment/
        checkpoint-7590/
          config.json
          trainer_state.json
          training_args.bin
          tokenizer.json
          tokenizer_config.json
          scaler.pt
          scheduler.pt
          model.safetensors
          optimizer.pt
          rng_state.pth


In [22]:
# ═══════════════════════════════════════════════════════
# LOAD MODEL + TOKENIZER
# What happens here:
#   1. BanglaBLIP built fresh (downloads ViT + BanglaT5)
#   2. LoRA structure applied (same config as training)
#   3. Saved weights loaded → overrides qformer, projection, LoRA weights
#   4. Tokenizer loaded from checkpoint
# ═══════════════════════════════════════════════════════

print("Step 1/3 — Building model architecture...")
model = BanglaBLIP(
    blip_pretrained=BLIP_CHECKPOINT,
    bangla_lm_pretrained=BANGLA_MODEL_ID,
    load_8bit="4bit",
    freeze_vit=True,
    freeze_qformer=False,
    freeze_lm=False,
    freeze_projection=False,
    use_lora=True,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
)

print("\nStep 2/3 — Loading saved checkpoint weights...")
import safetensors.torch as st
import os

safetensors_path = os.path.join(CHECKPOINT_DIR, "model.safetensors")
bin_path         = os.path.join(CHECKPOINT_DIR, "pytorch_model.bin")

if os.path.exists(safetensors_path):
    saved_weights = st.load_file(safetensors_path)
elif os.path.exists(bin_path):
    saved_weights = torch.load(bin_path, map_location="cpu")
else:
    raise FileNotFoundError(f"No model file found in {CHECKPOINT_DIR}")

missing, unexpected = model.load_state_dict(saved_weights, strict=False)
print(f"  Loaded weights — missing keys: {len(missing)} | unexpected: {len(unexpected)}")
if missing:
    print(f"  Missing (first 5): {missing[:5]}")

print("\nStep 3/3 — Loading tokenizer...")
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR, use_fast=False)

print("\nLoading BLIP2 processor (for image preprocessing)...")
processor = Blip2Processor.from_pretrained(BLIP_CHECKPOINT)

# Move all non-quantized components to GPU
model.vision_model        = model.vision_model.to(DEVICE).half()
model.qformer             = model.qformer.to(DEVICE).half()
model.query_tokens = torch.nn.Parameter(model.query_tokens.data.to(DEVICE).half())
model.language_projection = model.language_projection.to(DEVICE).half()
model.language_projection_ln = model.language_projection_ln.to(DEVICE).half()

model.eval()
print("\nModel ready for inference.")


Step 1/3 — Building model architecture...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1289 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie language_model.shared.weight to language_model.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Step 2/3 — Loading saved checkpoint weights...
  Loaded weights — missing keys: 1190 | unexpected: 0
  Missing (first 5): ['vision_model.embeddings.class_embedding', 'vision_model.embeddings.position_embedding', 'vision_model.embeddings.patch_embedding.weight', 'vision_model.embeddings.patch_embedding.bias', 'vision_model.encoder.layers.0.self_attn.qkv.weight']

Step 3/3 — Loading tokenizer...

Loading BLIP2 processor (for image preprocessing)...

Model ready for inference.


In [23]:
# ═══════════════════════════════════════════════════════
# LOAD TEST IMAGES
# Uses the same 80/10/10 split logic as training
# so we infer only on unseen test images
# ═══════════════════════════════════════════════════════

import pandas as pd
from sklearn.model_selection import train_test_split

def load_test_images(captions_csv, seed=42):
    df = pd.read_csv(captions_csv)
    df.columns = df.columns.str.strip().str.lower()

    captions_dict = {}
    for _, row in df.iterrows():
        cap_id   = str(row['caption_id']).strip()
        caption  = bangla_postprocess(str(row['bengali_caption']).strip())
        filename = cap_id.split('#')[0].strip()
        if not filename.endswith('.jpg'):
            filename += '.jpg'
        captions_dict.setdefault(filename, []).append(caption)

    all_images = sorted(captions_dict.keys())
    _, temp    = train_test_split(all_images, test_size=0.2,  random_state=seed)
    _, test    = train_test_split(temp,       test_size=0.5,  random_state=seed)

    print(f"Test set: {len(test)} images")
    return test, captions_dict

test_images, captions_dict = load_test_images(CAPTIONS_CSV)

if NUM_SAMPLES:
    import random
    random.seed(42)
    test_images = random.sample(test_images, min(NUM_SAMPLES, len(test_images)))
    print(f"Using {len(test_images)} samples for inference")


Test set: 810 images
Using 50 samples for inference


In [24]:
# ═══════════════════════════════════════════════════════
# RUN INFERENCE
# For each image:
#   1. Load & preprocess image (224×224, normalized)
#   2. Prompt: "বাংলায় ক্যাপশন:"
#   3. Generate caption with beam search
#   4. Post-process Bengali text
# ═══════════════════════════════════════════════════════

import csv
from tqdm import tqdm

PROMPT = "বাংলায় ক্যাপশন:"

results = []

print(f"Running inference on {len(test_images)} images...\n")

for filename in tqdm(test_images):
    img_path = os.path.join(IMAGES_DIR, filename)

    # Load image
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception:
        print(f"  Skipping missing image: {filename}")
        continue

    # Preprocess
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(DEVICE).half()
    model.vision_model = model.vision_model.to(DEVICE)

    # Tokenize prompt
    text_enc = tokenizer(
        PROMPT, return_tensors="pt",
        padding="max_length", max_length=128, truncation=True
    )
    input_ids      = text_enc["input_ids"].to(DEVICE)
    attention_mask = text_enc["attention_mask"].to(DEVICE)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_GEN_LENGTH,
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
            no_repeat_ngram_size=3,
            repetition_penalty=REPETITION_PENALTY,
            do_sample=False,
        )

    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated = bangla_postprocess(generated)

    references = captions_dict.get(filename, [])

    results.append({
        "filename":           filename,
        "generated_caption":  generated,
        "reference_captions": " ||| ".join(references),
    })

print(f"\nDone. Generated {len(results)} captions.")

# Show first 5
print("\n── Sample outputs ──────────────────────────────")
for r in results[:5]:
    print(f"\nImage : {r['filename']}")
    print(f"Generated : {r['generated_caption']}")
    print(f"Reference : {r['reference_captions'].split(' ||| ')[0]}")


Running inference on 50 images...



100%|██████████| 50/50 [02:35<00:00,  3.10s/it]


Done. Generated 50 captions.

── Sample outputs ──────────────────────────────

Image : 3648988742_888a16f600.jpg
Generated : বাংলাপিডিয়া থেকে। In Bengali: Bangladeshi.
Reference : একটি মেয়ে ক্যামেরা ধরে আছে

Image : 539705321_99406e5820.jpg
Generated : বাংলাপিডিয়া হতে সংকলিত নিবন্ধ। In Bengali: Input graphs. কিন্তু তিনি তা করেননি। But he did not.
Reference : সাদা রঙের জার্সি পরিহিত শিশুর সাথে বল দখলের লড়াইয়ে লাল রঙের জার্সি পরিহিত শিশু মাটিতে পড়ে যাচ্ছে

Image : 3660361818_e05367693f.jpg
Generated : The Bible. In Bengali: The বাইবেল। কিন্তু, তিনি তা করতে পারেননি। But he did not.
Reference : এক ব্যাক্তি স্কেটবোর্ডিং ট্রিক করছে

Image : 2436081047_bca044c1d3.jpg
Generated : In Bengal: Remarks: "The Rest": The Restে (ইংরেজি ভাষায়)
Reference : মেয়েটি দোলনা থেকে মেঝের দিকে লাফ দিয়েছে

Image : 385835044_4aa11f6990.jpg
Generated : bangla calculations Head to the front door. প্রধান দরজার দিকে এগোও।
Reference : বাদামি জ্যাকেট এবং নীল জিনস পরিহিত একজন মেয়ে কানে সাদা হেডফোন লাগিয়ে হে

In [25]:
# ═══════════════════════════════════════════════════════
# SAVE RESULTS TO CSV
# ═══════════════════════════════════════════════════════

import csv

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "generated_caption", "reference_captions"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved {len(results)} rows → {OUTPUT_CSV}")


Saved 50 rows → /kaggle/working/inference_results.csv


In [26]:
# ═══════════════════════════════════════════════════════
# OPTIONAL: BLEU + CIDEr + METEOR on inference results
# ═══════════════════════════════════════════════════════

import math, numpy as np
from collections import defaultdict
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

smoothie     = SmoothingFunction().method1
predictions  = []
references   = []

for r in results:
    hyp  = r["generated_caption"].split()
    refs = [ref.split() for ref in r["reference_captions"].split(" ||| ")]
    predictions.append(hyp)
    references.append(refs)

bleu1 = corpus_bleu(references, predictions, weights=(1,0,0,0),            smoothing_function=smoothie)
bleu2 = corpus_bleu(references, predictions, weights=(0.5,0.5,0,0),         smoothing_function=smoothie)
bleu3 = corpus_bleu(references, predictions, weights=(0.33,0.33,0.33,0),    smoothing_function=smoothie)
bleu4 = corpus_bleu(references, predictions, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie)

meteor = float(np.mean([
    meteor_score(refs, hyp) for hyp, refs in zip(predictions, references)
]))

print("=" * 45)
print("  INFERENCE EVALUATION RESULTS")
print("=" * 45)
print(f"  BLEU-1  : {bleu1:.4f}")
print(f"  BLEU-2  : {bleu2:.4f}")
print(f"  BLEU-3  : {bleu3:.4f}")
print(f"  BLEU-4  : {bleu4:.4f}")
print(f"  METEOR  : {meteor:.4f}")
print("=" * 45)


  INFERENCE EVALUATION RESULTS
  BLEU-1  : 0.0059
  BLEU-2  : 0.0011
  BLEU-3  : 0.0007
  BLEU-4  : 0.0005
  METEOR  : 0.0039
